In [1]:
import gc
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split



In [2]:
feat_cols = ["R", "C", "time_step", "u_in", "u_out"]
dtype_spec = {
    "R": "int8",
    "C": "int8",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "int8",
    "pressure": "float32",
}
train_path = "../input/ventilator-pressure-prediction/train.csv"
test_path = "../input/ventilator-pressure-prediction/test.csv"
sample_sub_path = "../input/ventilator-pressure-prediction/sample_submission.csv"

train_df = pd.read_csv(
    train_path,
    usecols=feat_cols + ["pressure"],
    dtype=dtype_spec,
    engine="c",
    memory_map=True,
)
test_df = pd.read_csv(
    test_path,
    usecols=feat_cols + ["id"],
    dtype={**dtype_spec, "id": "int32"},
    engine="c",
    memory_map=True,
)
sample_sub = pd.read_csv(sample_sub_path)



In [3]:
# Convert to numpy once; keep float32 to limit memory bandwidth.
X = train_df[feat_cols].to_numpy(dtype=np.float32, copy=False)
y = train_df["pressure"].to_numpy(dtype=np.float32, copy=False)

del train_df
gc.collect()

# Speed‑up: fewer trees and a smaller per‑tree sample while keeping the same model type.
rf = RandomForestRegressor(
    n_estimators=150,  # reduced from 250 to cut training time
    max_depth=15,
    min_samples_leaf=2,
    max_features="sqrt",
    max_samples=0.60,  # smaller subsample per tree (was 0.85)
    n_jobs=-1,
    random_state=42,
)

rf.fit(X, y)



RandomForestRegressor(max_depth=15, max_features='sqrt', max_samples=0.6,
                      min_samples_leaf=2, n_estimators=150, n_jobs=-1,
                      random_state=42)

In [4]:
test_features = test_df[feat_cols].to_numpy(dtype=np.float32, copy=False)
test_pred = rf.predict(test_features)

submission = pd.DataFrame({"id": test_df["id"], "pressure": test_pred})
submission = submission[["id", "pressure"]]
submission.to_csv("submission.csv", index=False)

submission.head()

,id,pressure
0,1,5.574630
1,2,5.559700
2,3,7.201115
3,4,13.142775
4,5,16.165559
